### Loading Data

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import qnt.data as qndata
import qnt.ta as qnta

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

close = data.sel(field="close")
vol = data.sel(field="vol")
is_liquid = data.sel(field="is_liquid")

close = xr.where(is_liquid == 1, close, np.nan)
vol = xr.where(is_liquid == 1, vol, np.nan)

print(close.shape)

100% (15262844 of 15262844) |############| Elapsed Time: 0:00:00 Time:  0:00:00
(4179, 76)

### Return Features

In [ ]:
# ==========================================
# Return Features
# ==========================================

ret_1d = close / close.shift(time=1) - 1

ret_5d = close / close.shift(time=5) - 1

ret_10d = close / close.shift(time=10) - 1

ret_21d = close / close.shift(time=21) - 1

ret_63d = close / close.shift(time=63) - 1

### Volatility Features

In [ ]:
# ==========================================
# Volatility Features
# ==========================================

vol_10d = qnta.std(ret_1d, 10)

vol_20d = qnta.std(ret_1d, 20)

vol_60d = qnta.std(ret_1d, 60)

### Volume Features

In [ ]:
# ==========================================
# Volume Features
# ==========================================

vol_ma20 = qnta.sma(vol, 20)

vol_ma60 = qnta.sma(vol, 60)

volume_ratio_20d = (
    vol /
    (vol_ma20 + 1e-6)
)

volume_ratio_60d = (
    vol /
    (vol_ma60 + 1e-6)
)

### Trend Features

In [ ]:
# ==========================================
# Trend Features
# ==========================================

sma20 = qnta.sma(close, 20)

sma50 = qnta.sma(close, 50)

sma200 = qnta.sma(close, 200)

price_vs_sma20 = (
    close /
    (sma20 + 1e-6)
)

price_vs_sma50 = (
    close /
    (sma50 + 1e-6)
)

price_vs_sma200 = (
    close /
    (sma200 + 1e-6)
)

### RSI

In [ ]:
# ==========================================
# RSI
# ==========================================

rsi14 = qnta.rsi(close, 14)

### MACD

In [ ]:
# ==========================================
# MACD
# ==========================================

ema12 = qnta.ema(close, 12)

ema26 = qnta.ema(close, 26)

macd = ema12 - ema26

macd_signal = qnta.ema(macd, 9)

macd_hist = macd - macd_signal

### Bollinger Features

In [ ]:
# ==========================================
# Bollinger Bands
# ==========================================

bb_mid = qnta.sma(close, 20)

bb_std = qnta.std(close, 20)

bb_upper = bb_mid + 2 * bb_std

bb_lower = bb_mid - 2 * bb_std

bollinger_position = (
    (close - bb_lower)
    /
    (bb_upper - bb_lower + 1e-6)
)

bollinger_width = (
    (bb_upper - bb_lower)
    /
    (bb_mid + 1e-6)
)

### Feature Dictionary

In [ ]:
features = {

    "ret_1d": ret_1d,
    "ret_5d": ret_5d,
    "ret_10d": ret_10d,
    "ret_21d": ret_21d,
    "ret_63d": ret_63d,

    "vol_10d": vol_10d,
    "vol_20d": vol_20d,
    "vol_60d": vol_60d,

    "volume_ratio_20d": volume_ratio_20d,
    "volume_ratio_60d": volume_ratio_60d,

    "price_vs_sma20": price_vs_sma20,
    "price_vs_sma50": price_vs_sma50,
    "price_vs_sma200": price_vs_sma200,

    "rsi14": rsi14,

    "macd": macd,
    "macd_signal": macd_signal,
    "macd_hist": macd_hist,

    "bollinger_position": bollinger_position,
    "bollinger_width": bollinger_width
}

print(
    f"Total Features Created: {len(features)}"
)

Total Features Created: 19

### Feature Validation

In [ ]:
# ==========================================
# Feature Validation
# ==========================================

print("Latest values")

for name, feature in features.items():

    try:

        latest = feature.isel(time=-1)

        print("\n", name)
        print(latest.to_pandas().describe())

    except:

        print(f"Problem with {name}")

Latest values

 ret_1d
count    10.000000
mean     -0.022954
std       0.026319
min      -0.077634
25%      -0.033370
50%      -0.016630
75%      -0.006933
max       0.014427
Name: cryptodaily, dtype: float64

 ret_5d
count    10.000000
mean      0.000197
std       0.041502
min      -0.108081
25%      -0.002938
50%       0.004730
75%       0.021657
max       0.047372
Name: cryptodaily, dtype: float64

 ret_10d
count    9.000000
mean    -0.167319
std      0.066161
min     -0.260918
25%     -0.191504
50%     -0.174233
75%     -0.164707
max     -0.046465
Name: cryptodaily, dtype: float64

 ret_21d
count    9.000000
mean    -0.155026
std      0.084922
min     -0.266612
25%     -0.206785
50%     -0.197285
75%     -0.096419
max     -0.029739
Name: cryptodaily, dtype: float64

 ret_63d
count    9.000000
mean    -0.068270
std      0.189514
min     -0.260184
25%     -0.182815
50%     -0.103515
75%     -0.024137
max      0.373757
Name: cryptodaily, dtype: float64

 vol_10d
count    9.000000
mean     0.035230
std      0.016794
min      0.014519
25%      0.027842
50%      0.033630
75%      0.040682
max      0.070446
dtype: float64

 vol_20d
count    9.000000
mean     0.030697
std      0.015550
min      0.011545
25%      0.022531
50%      0.028713
75%      0.034585
max      0.065999
dtype: float64

 vol_60d
count    9.000000
mean     0.024951
std      0.012279
min      0.010177
25%      0.019931
50%      0.024447
75%      0.027431
max      0.053034
dtype: float64

 volume_ratio_20d
count    9.000000
mean     0.701387
std      0.141392
min      0.468759
25%      0.587063
50%      0.732488
75%      0.784703
max      0.891342
dtype: float64

 volume_ratio_60d
count    9.000000
mean     0.763441
std      0.330620
min      0.427030
25%      0.604746
50%      0.698671
75%      0.782079
max      1.573457
dtype: float64

 price_vs_sma20
count    9.000000
mean     0.889406
std      0.043567
min      0.831946
25%      0.858393
50%      0.878911
75%      0.920668
max      0.968567
dtype: float64

 price_vs_sma50
count    9.000000
mean     0.868917
std      0.096469
min      0.761522
25%      0.815110
50%      0.819294
75%      0.936479
max      1.043481
dtype: float64

 price_vs_sma200
count    8.000000
mean     0.850858
std      0.269226
min      0.624699
25%      0.681266
50%      0.767290
75%      0.872089
max      1.434206
dtype: float64

 rsi14
count     9.000000
mean     31.585390
std       7.315287
min      23.876852
25%      25.118478
50%      28.300968
75%      37.953466
max      44.664843
dtype: float64

 macd
count    9.000000
mean    -0.039689
std      0.037090
min     -0.078695
25%     -0.060088
50%     -0.055856
75%     -0.021920
max      0.037578
dtype: float64

 macd_signal
count    9.000000
mean    -0.024834
std      0.042155
min     -0.063449
25%     -0.047769
50%     -0.044585
75%     -0.012368
max      0.073752
dtype: float64

 macd_hist
count    9.000000
mean    -0.014855
std      0.008798
min     -0.036173
25%     -0.013983
50%     -0.012374
75%     -0.011271
max     -0.005270
dtype: float64

 bollinger_position
count    9.000000
mean     0.151317
std      0.033283
min      0.107226
25%      0.130838
50%      0.149973
75%      0.179228
max      0.192012
dtype: float64

 bollinger_width
count    9.000000
mean     0.311607
std      0.108783
min      0.099870
25%      0.257580
50%      0.330386
75%      0.383585
max      0.455231
dtype: float64

### Correlation Analysis

In [ ]:
# ==========================================
# Correlation Analysis
# ==========================================

latest_features = pd.DataFrame()

for name, feature in features.items():

    try:

        latest_features[name] = (
            feature
            .isel(time=-1)
            .to_pandas()
        )

    except:
        pass

ret_1d	ret_5d	ret_10d	ret_21d	ret_63d	vol_10d	vol_20d	vol_60d	volume_ratio_20d	volume_ratio_60d	price_vs_sma20	price_vs_sma50	price_vs_sma200	rsi14	macd	macd_signal	macd_hist	bollinger_position	bollinger_width
ret_1d	1.00	0.54	0.81	-0.14	-0.59	-0.89	-0.89	-0.92	-0.76	-0.84	0.68	-0.30	-0.64	-0.35	-0.44	-0.57	0.92	0.82	-0.57
ret_5d	0.54	1.00	0.45	-0.53	-0.83	-0.68	-0.72	-0.76	-0.45	-0.90	0.31	-0.66	-0.84	-0.62	-0.76	-0.85	0.86	0.52	-0.20
ret_10d	0.81	0.45	1.00	0.31	-0.17	-0.90	-0.84	-0.87	-0.79	-0.62	0.92	0.17	-0.18	0.05	0.03	-0.14	0.81	0.68	-0.90
ret_21d	-0.14	-0.53	0.31	1.00	0.84	0.00	0.23	0.14	-0.40	0.37	0.58	0.98	0.89	0.95	0.95	0.87	-0.21	0.25	-0.64
ret_63d	-0.59	-0.83	-0.17	0.84	1.00	0.45	0.62	0.58	0.04	0.70	0.09	0.92	0.97	0.87	0.96	0.99	-0.67	-0.14	-0.16
vol_10d	-0.89	-0.68	-0.90	0.00	0.45	1.00	0.95	0.98	0.70	0.79	-0.79	0.13	0.48	0.23	0.28	0.44	-0.93	-0.72	0.72
vol_20d	-0.89	-0.72	-0.84	0.23	0.62	0.95	1.00	0.98	0.57	0.83	-0.63	0.35	0.65	0.46	0.48	0.62	-0.94	-0.56	0.57
vol_60d	-0.92	-0.76	-0.87	0.14	0.58	0.98	0.98	1.00	0.65	0.83	-0.71	0.27	0.61	0.36	0.41	0.56	-0.97	-0.67	0.65
volume_ratio_20d	-0.76	-0.45	-0.79	-0.40	0.04	0.70	0.57	0.65	1.00	0.65	-0.84	-0.25	0.09	-0.26	-0.11	0.03	-0.62	-0.84	0.78
volume_ratio_60d	-0.84	-0.90	-0.62	0.37	0.70	0.79	0.83	0.83	0.65	1.00	-0.47	0.50	0.75	0.48	0.61	0.72	-0.87	-0.54	0.39
price_vs_sma20	0.68	0.31	0.92	0.58	0.09	-0.79	-0.63	-0.71	-0.84	-0.47	1.00	0.43	0.09	0.37	0.30	0.12	0.67	0.75	-0.98
price_vs_sma50	-0.30	-0.66	0.17	0.98	0.92	0.13	0.35	0.27	-0.25	0.50	0.43	1.00	0.94	0.94	0.99	0.95	-0.37	0.14	-0.50
price_vs_sma200	-0.64	-0.84	-0.18	0.89	0.97	0.48	0.65	0.61	0.09	0.75	0.09	0.94	1.00	0.84	0.96	0.97	-0.72	-0.14	-0.20
rsi14	-0.35	-0.62	0.05	0.95	0.87	0.23	0.46	0.36	-0.26	0.48	0.37	0.94	0.84	1.00	0.95	0.92	-0.41	0.07	-0.44
macd	-0.44	-0.76	0.03	0.95	0.96	0.28	0.48	0.41	-0.11	0.61	0.30	0.99	0.96	0.95	1.00	0.98	-0.50	-0.01	-0.38
macd_signal	-0.57	-0.85	-0.14	0.87	0.99	0.44	0.62	0.56	0.03	0.72	0.12	0.95	0.97	0.92	0.98	1.00	-0.65	-0.14	-0.21
macd_hist	0.92	0.86	0.81	-0.21	-0.67	-0.93	-0.94	-0.97	-0.62	-0.87	0.67	-0.37	-0.72	-0.41	-0.50	-0.65	1.00	0.62	-0.61
bollinger_position	0.82	0.52	0.68	0.25	-0.14	-0.72	-0.56	-0.67	-0.84	-0.54	0.75	0.14	-0.14	0.07	-0.01	-0.14	0.62	1.00	-0.60
bollinger_width	-0.57	-0.20	-0.90	-0.64	-0.16	0.72	0.57	0.65	0.78	0.39	-0.98	-0.50	-0.20	-0.44	-0.38	-0.21	-0.61	-0.60	1.00

In [ ]:
corr.loc[
    [
        "ret_21d",
        "ret_63d",
        "price_vs_sma200",
        "rsi14",
        "macd",
        "bollinger_position"
    ],
    :
]

### Findings - 


- ret_21d, RSI, MACD and price_vs_sma200 form a trend cluster.
- Volume features show lower correlation with trend indicators.
- Volatility features form a separate risk cluster.
- Most technical indicators are different representations of trend.

## Feature IC Analysis

In [ ]:
# ==========================================
# Feature IC Analysis
# ==========================================

future_return = (
    close.shift(time=-21)
    / close
    - 1
)

In [ ]:
def cross_sectional_ic(feature, target):

    ics = []

    for t in feature.time.values:

        try:

            x = feature.sel(time=t)
            y = target.sel(time=t)

            mask = (
                np.isfinite(x)
                &
                np.isfinite(y)
            )

            x = x.where(mask, drop=True)
            y = y.where(mask, drop=True)

            if len(x.asset) < 10:
                continue

            ic = xr.corr(
                x,
                y,
                dim="asset"
            )

            if np.isfinite(ic):
                ics.append(float(ic))

        except:
            pass

    return np.mean(ics)

In [ ]:
selected_features = {

    "trend": price_vs_sma200,

    "volume": volume_ratio_20d,

    "volatility": vol_20d,

    "bollinger_width": bollinger_width,

    "short_term_return": ret_5d

}

In [ ]:
btc = data.sel(
    field="close",
    asset="BTC"
)

btc_ret_21d = (
    btc /
    btc.shift(time=21)
    - 1
)

btc_rs = (
    ret_21d
    - btc_ret_21d
)

selected_features["btc_rs"] = btc_rs

In [ ]:
results = []

for name, feature in selected_features.items():

    ic = cross_sectional_ic(
        feature,
        future_return
    )

    results.append(
        [name, ic]
    )

ic_table = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "IC"
    ]
)

ic_table = ic_table.sort_values(
    "IC",
    ascending=False
)

print(ic_table)

Feature IC Results

BTC RS              0.0094
Volume              0.0081
Short-Term Return   0.0078
Trend              -0.0134
Bollinger Width    -0.1080
Volatility         -0.1388

Key Findings:

- BTC Relative Strength produced the highest positive IC.
- Volume contributed a small positive signal.
- Traditional trend indicators were weak predictors.
- High volatility predicted lower future returns.
- Wide Bollinger Bands predicted lower future returns.
- BTC Relative Strength remained the strongest overall factor discovered during the research process.